In [199]:
import pandas as pd
import numpy as np
import random
from collections import Counter

# PROLOGUE PARAMETER

In [200]:
POPULASI = 50
VIOLATION_COST = 100

In [201]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')

# 1. join relasi

In [202]:
# guru_df = relasi_guru_mapel_df = mapel-df
join = (
    relasi_guru_mapel_df
    .merge(guru_df, on="guru_id", how="left")
    .merge(mapel_df, on="mapel_id", how="left")
)

# penambahan kolom "total"
join["total"] = (
    join.groupby("guru_id")["durasi"]
    .transform("sum")
) 
# join.head(3)

In [203]:
# join.count()

# 2. Menggunakan Numerik

### konversi hari ke numerik

In [204]:
# dictionary hari -> numerik
mapping_hari = {
    "Senin": 1,
    "Selasa": 2,
    "Rabu": 3,
    "Kamis": 4,
    "Jumat": 5,
}

### mengambil kolom penting

In [205]:
# mengambil kolom numerik
relasi = join[[
    "guru_id",
    "mapel_id",
    "jam_per_minggu",
    "tingkatan",
    "durasi",
    "total",
    "MGMP"
]].copy()

# melakukan konversi hari ke numerik pakai mapping
relasi["MGMP"] = relasi["MGMP"].map(mapping_hari)
# relasi.info()

### Dict Mapel dan jam

In [206]:
mapel_jam = dict(
    zip(mapel_df["mapel_id"], mapel_df["jam_per_minggu"])
)
# mapel_jam

### mapping hari dan jumlah slot

In [207]:
# mapping jumlah slot per hari
slot_per_hari = {
    1 : 8, # senin
    2 : 8, # Selasa
    3 : 8, # rabu
    4 : 7, # kamis
    5 : 5, # jumat
}

### count kelas

In [208]:
# mengambil total kelas (27)
total_kelas = kelas_df["kelas_id"].count()
# total_kelas

### Batas siang
digunakan untuk mapel PJOK (8)

In [209]:
batas_siang = {
    0: 5,  # Senin  → index 6,7 dilarang
    1: 5,  # Selasa → index 6,7 dilarang
    2: 4,  # Rabu   → index 5,6,7 dilarang
    3: 5,  # Kamis  → index 6 dilarang
    4: 4   # Jumat  → index 5 dilarang
}

### batas MGMP mapel

In [210]:
batas_mgmp = {
    0: 2,  # Senin
    1: 2,  # Selasa
    2: 2,  # Rabu
    3: 2,  # Kamis
    4: 1   # Jumat
}

# 3. Groupping guru  
mapel_id dan tingkatan

In [211]:
guru_by_mapel = (
    relasi
    .groupby(["mapel_id", "tingkatan"])["guru_id"]
    .apply(list)
    .to_dict()
)
# guru_by_mapel

# 4. Daftar Variabel

variabel variabel yang telah diproses dan siap digunakan
| Nama Variabel| Keterangan Singkat |
|--------------|-------------------|
| `guru_df` |  Original guru |
| `kelas_df` |  Original kelas |
| `mapel_df` |  Original mapel |
| `relasi_guru_mapel_df` |  Original relasi |
| `slot_df` |  Original slot |
| `relasi` |  Relasi guru & mapel (93 rows) |
| `guru_by_mapel` |  Guru groupping |
| `total_kelas`| 27 (int)|
|`slot_per_hari`| dictionary |
| `mapel_jam`| dictionary|
| `batas_siang`| dictionary |


# 5. Fungsi Inisialisasi Individu

In [212]:
def individuConstruct(slot_per_hari, total_kelas, guru_by_mapel, relasi):

    mapel_id = list(range(1, 14))     # 13 mapel
    tingkatan = relasi["tingkatan"].iloc[0]

    # preprocess guru per mapel
    guru_mapel_list = {
        m: guru_by_mapel.get((m, tingkatan), [])
        for m in mapel_id
    }

    individu = []

    for _ in range(total_kelas):

        kelas = []

        # ===== GEN MAPEL PER HARI =====
        for hari in sorted(slot_per_hari.keys()):  # 1..5
            jumlah_slot = slot_per_hari[hari]
            gen_hari = [random.choice(mapel_id) for _ in range(jumlah_slot)]
            kelas.append(gen_hari)

        # ===== GEN GURU (13 MAPEL) =====
        gen_guru = [
            random.choice(guru_mapel_list[m]) if guru_mapel_list[m] else 0
            for m in mapel_id
        ]

        kelas.append(gen_guru)

        individu.append(kelas)

    return individu


In [213]:
individu = individuConstruct(slot_per_hari, total_kelas, guru_by_mapel, relasi)
# individu

In [214]:
# individu

In [215]:
# individu[0]

In [216]:
populasi = []
for i in range(POPULASI):
    individu = individuConstruct(slot_per_hari, total_kelas, guru_by_mapel, relasi)
    populasi.append(individu)

In [217]:
# populasi

In [218]:
# individu =   
# [ # ini 1 individu  
#         [ # ini 1 kelas  
#                 [(m1,1), (m2,1), (m3,0), (m4, 0), (m5, 1), (m6, 1), (m7, 1), (m8, 1)],   # Senin (8)  
#                 [(m1, 0), (m2, 0), (m3, 1), (m4, 1), (m5, 1), (m6, 1), (m7, 1), (m8, 1)],   # Selasa (8)  
#                 [(m1, 0), (m2, 0 ), (m3, 1), (m4, 0), (m5, 1), (m6, 1), (m7, 0), (m8, 0)],   # Rabu (8)  
#                 [(m1, 0), (m2, 0), (m3, 1), (m4, 1), (m5, 1), (m6, 1), (m7, 1)],       # Kamis (7)  
#                 [(m1, 0), (m2, 1), (m3, 0), (m4, 0), (m5, 0)],                # Jumat (5)  
#                 [g1, g2, g3, g4, g5, sampai g13]        # guru pengajar (13)  
#         ],  
#         [  
#                 # kelas lain  
#         ]  
# ]  

# 6. Evaluasi Individu

### 6.1. Guru hanya boleh mengajar 1 slot waktu, tidak boleh lebih

In [219]:
def guru_bentrok(individu, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    slot_harian = individu[0][:-1]

    for hari_id, hari in enumerate(slot_harian):
        for jam_id in range(len(hari)):
            guru_used = set()

            for kelas in individu:
                mapel = kelas[hari_id][jam_id]

                if mapel == 0:
                    continue

                guru = kelas[-1][mapel - 1]

                if guru in guru_used:
                    pelanggaran += 1
                    cost += VIOLATION_COST
                else:
                    guru_used.add(guru)

    return pelanggaran, cost

In [220]:
for individu in populasi:
    pelanggaran, cost = guru_bentrok(individu, VIOLATION_COST)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  369
Cost:  36900
Hard Constraint:  381
Cost:  38100
Hard Constraint:  363
Cost:  36300
Hard Constraint:  386
Cost:  38600
Hard Constraint:  373
Cost:  37300
Hard Constraint:  387
Cost:  38700
Hard Constraint:  366
Cost:  36600
Hard Constraint:  363
Cost:  36300
Hard Constraint:  375
Cost:  37500
Hard Constraint:  350
Cost:  35000
Hard Constraint:  360
Cost:  36000
Hard Constraint:  373
Cost:  37300
Hard Constraint:  384
Cost:  38400
Hard Constraint:  377
Cost:  37700
Hard Constraint:  388
Cost:  38800
Hard Constraint:  368
Cost:  36800
Hard Constraint:  362
Cost:  36200
Hard Constraint:  382
Cost:  38200
Hard Constraint:  368
Cost:  36800
Hard Constraint:  384
Cost:  38400
Hard Constraint:  383
Cost:  38300
Hard Constraint:  360
Cost:  36000
Hard Constraint:  390
Cost:  39000
Hard Constraint:  357
Cost:  35700
Hard Constraint:  373
Cost:  37300
Hard Constraint:  381
Cost:  38100
Hard Constraint:  387
Cost:  38700
Hard Constraint:  362
Cost:  36200
Hard Constraint:  36

### 6.2. konfigurasi mapel  
2 jam -> 1 hari  
3 jam -> 1 hari    
4 jam -> 2 hari berbeda [2,2]  
5 jam -> 2 hari berbeda [2,3]

In [221]:
def konfigurasi_mapel(individu, mapel_jam, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1]  # senin–jumat

        mapel_hari = {}

        for hari_id, hari in enumerate(slot_harian):
            for slot_id, mapel in enumerate(hari):
                if mapel not in mapel_hari:
                    mapel_hari[mapel] = {}
                if hari_id not in mapel_hari[mapel]:
                    mapel_hari[mapel][hari_id] = []
                mapel_hari[mapel][hari_id].append(slot_id)

        # evaluasi per mapel
        for mapel_id, distribusi_hari in mapel_hari.items():
            total_jam = mapel_jam.get(mapel_id, 0)

            # rule jumlah hari
            hari_terpakai = [len(v) for v in distribusi_hari.values()]

            if total_jam in (2, 3):
                # mapel dengan 2 , 3 jam per minggu
                if len(distribusi_hari) != 1:
                    pelanggaran += 1
                    cost += VIOLATION_COST

            elif total_jam == 4:
                # mapel dengan 4 jam per minggu
                if sorted(hari_terpakai) != [2, 2]:
                    pelanggaran += 1
                    cost += VIOLATION_COST

            elif total_jam == 5:
                # mapel dengan 5 jam per minggu
                if sorted(hari_terpakai) != [2, 3]:
                    pelanggaran += 1
                    cost += VIOLATION_COST

            # mapel harus pada slot yang berdekatan pada hari yang sama
            for slot_list in distribusi_hari.values():
                if len(slot_list) > 1:
                    slot_list = sorted(slot_list)
                    for i in range(len(slot_list) - 1):
                        if slot_list[i + 1] - slot_list[i] != 1:
                            pelanggaran += 1
                            cost += VIOLATION_COST
                            break

    return pelanggaran, cost

In [222]:
for individu in populasi:
    pelanggaran, cost = konfigurasi_mapel(individu,mapel_jam, VIOLATION_COST)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  396
Cost:  39600
Hard Constraint:  406
Cost:  40600
Hard Constraint:  410
Cost:  41000
Hard Constraint:  415
Cost:  41500
Hard Constraint:  403
Cost:  40300
Hard Constraint:  401
Cost:  40100
Hard Constraint:  408
Cost:  40800
Hard Constraint:  414
Cost:  41400
Hard Constraint:  406
Cost:  40600
Hard Constraint:  407
Cost:  40700
Hard Constraint:  405
Cost:  40500
Hard Constraint:  404
Cost:  40400
Hard Constraint:  410
Cost:  41000
Hard Constraint:  397
Cost:  39700
Hard Constraint:  396
Cost:  39600
Hard Constraint:  418
Cost:  41800
Hard Constraint:  419
Cost:  41900
Hard Constraint:  410
Cost:  41000
Hard Constraint:  400
Cost:  40000
Hard Constraint:  423
Cost:  42300
Hard Constraint:  410
Cost:  41000
Hard Constraint:  425
Cost:  42500
Hard Constraint:  409
Cost:  40900
Hard Constraint:  400
Cost:  40000
Hard Constraint:  410
Cost:  41000
Hard Constraint:  407
Cost:  40700
Hard Constraint:  405
Cost:  40500
Hard Constraint:  393
Cost:  39300
Hard Constraint:  41

### 6.3. Mapel PJOK

In [223]:
def mapel_pjok(individu, batas_siang, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1] # hari saja

        for hari_id, hari in enumerate(slot_harian):
            batas = batas_siang[hari_id]

            for slot_id, mapel in enumerate(hari):
                if mapel == 8 and slot_id > batas:
                    pelanggaran += 1
                    cost += VIOLATION_COST
                    
    return pelanggaran, cost

In [224]:
for individu in populasi:
    pelanggaran, cost = mapel_pjok(individu,batas_siang, VIOLATION_COST)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  15
Cost:  1500
Hard Constraint:  12
Cost:  1200
Hard Constraint:  14
Cost:  1400
Hard Constraint:  11
Cost:  1100
Hard Constraint:  14
Cost:  1400
Hard Constraint:  20
Cost:  2000
Hard Constraint:  16
Cost:  1600
Hard Constraint:  8
Cost:  800
Hard Constraint:  17
Cost:  1700
Hard Constraint:  24
Cost:  2400
Hard Constraint:  24
Cost:  2400
Hard Constraint:  13
Cost:  1300
Hard Constraint:  18
Cost:  1800
Hard Constraint:  11
Cost:  1100
Hard Constraint:  17
Cost:  1700
Hard Constraint:  12
Cost:  1200
Hard Constraint:  17
Cost:  1700
Hard Constraint:  20
Cost:  2000
Hard Constraint:  17
Cost:  1700
Hard Constraint:  17
Cost:  1700
Hard Constraint:  17
Cost:  1700
Hard Constraint:  10
Cost:  1000
Hard Constraint:  18
Cost:  1800
Hard Constraint:  10
Cost:  1000
Hard Constraint:  20
Cost:  2000
Hard Constraint:  13
Cost:  1300
Hard Constraint:  25
Cost:  2500
Hard Constraint:  21
Cost:  2100
Hard Constraint:  13
Cost:  1300
Hard Constraint:  23
Cost:  2300
Hard Constra

### 6.4. Durasi mengajar Guru

In [225]:
def durasi_guru(individu, relasi, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    # lookup (guru, mapel) -> durasi
    durasi_lookup = {
        (row.guru_id, row.mapel_id) : row.durasi
        for row in relasi.itertuples(index=False)
    }

    for kelas in individu:
        slot_harian = kelas[:-1]
        guru_mapel = kelas[-1]

    # menghitung total jam
    jam_guru_aktual = Counter()

    for hari in slot_harian:
        for mapel in hari:
            guru = guru_mapel[mapel - 1]
            jam_guru_aktual[(guru, mapel)] += 1

    for (guru, mapel), jam_aktual in jam_guru_aktual.items():
        durasi_wajib = durasi_lookup.get((guru, mapel))

        if durasi_wajib is None:
            # tidak valid di relasi.df
            pelanggaran += 1
            cost += VIOLATION_COST

        elif jam_aktual != durasi_wajib:
            pelanggaran += abs(jam_aktual - durasi_wajib)
            cost += VIOLATION_COST * abs(jam_aktual - durasi_wajib)

    return pelanggaran, cost

In [226]:
for individu in populasi:
    pelanggaran, cost = durasi_guru(individu, relasi, VIOLATION_COST)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  135
Cost:  13500
Hard Constraint:  115
Cost:  11500
Hard Constraint:  97
Cost:  9700
Hard Constraint:  132
Cost:  13200
Hard Constraint:  127
Cost:  12700
Hard Constraint:  133
Cost:  13300
Hard Constraint:  143
Cost:  14300
Hard Constraint:  104
Cost:  10400
Hard Constraint:  142
Cost:  14200
Hard Constraint:  104
Cost:  10400
Hard Constraint:  102
Cost:  10200
Hard Constraint:  122
Cost:  12200
Hard Constraint:  113
Cost:  11300
Hard Constraint:  149
Cost:  14900
Hard Constraint:  99
Cost:  9900
Hard Constraint:  128
Cost:  12800
Hard Constraint:  130
Cost:  13000
Hard Constraint:  143
Cost:  14300
Hard Constraint:  134
Cost:  13400
Hard Constraint:  162
Cost:  16200
Hard Constraint:  136
Cost:  13600
Hard Constraint:  128
Cost:  12800
Hard Constraint:  86
Cost:  8600
Hard Constraint:  64
Cost:  6400
Hard Constraint:  131
Cost:  13100
Hard Constraint:  127
Cost:  12700
Hard Constraint:  97
Cost:  9700
Hard Constraint:  96
Cost:  9600
Hard Constraint:  139
Cost:  139

### 6.5. Mapel sesuai dengan jam_per_minggu

In [227]:
def mapel_jam_per_minggu(individu, mapel_jam, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1]

        semua_slot = []
        for hari in slot_harian:
            semua_slot.extend(hari)

        hitung_mapel = Counter(semua_slot)

        for mapel_id, jam_wajib in mapel_jam.items():
            jam_aktual = hitung_mapel.get(mapel_id, 0)

            if jam_aktual != jam_wajib:
                diff = abs(jam_aktual - jam_wajib)

                pelanggaran += diff
                cost += diff * VIOLATION_COST
                
    return pelanggaran, cost

In [228]:
for individu in populasi:
    pelanggaran, cost = mapel_jam_per_minggu(individu, mapel_jam, VIOLATION_COST)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  466
Cost:  46600
Hard Constraint:  546
Cost:  54600
Hard Constraint:  546
Cost:  54600
Hard Constraint:  526
Cost:  52600
Hard Constraint:  516
Cost:  51600
Hard Constraint:  532
Cost:  53200
Hard Constraint:  530
Cost:  53000
Hard Constraint:  514
Cost:  51400
Hard Constraint:  536
Cost:  53600
Hard Constraint:  508
Cost:  50800
Hard Constraint:  496
Cost:  49600
Hard Constraint:  500
Cost:  50000
Hard Constraint:  506
Cost:  50600
Hard Constraint:  486
Cost:  48600
Hard Constraint:  520
Cost:  52000
Hard Constraint:  530
Cost:  53000
Hard Constraint:  488
Cost:  48800
Hard Constraint:  510
Cost:  51000
Hard Constraint:  508
Cost:  50800
Hard Constraint:  502
Cost:  50200
Hard Constraint:  526
Cost:  52600
Hard Constraint:  522
Cost:  52200
Hard Constraint:  526
Cost:  52600
Hard Constraint:  476
Cost:  47600
Hard Constraint:  532
Cost:  53200
Hard Constraint:  494
Cost:  49400
Hard Constraint:  486
Cost:  48600
Hard Constraint:  520
Cost:  52000
Hard Constraint:  53

### 6.6. MGMP

In [229]:
def build_mapel_mgmp(relasi):
    return {
        row.mapel_id: row.MGMP - 1
        for row in relasi.itertuples(index=False)
        if row.MGMP is not None
    }

In [230]:
def mgmp_slot(individu, relasi, batas_mgmp, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    mapel_mgmp = build_mapel_mgmp(relasi)

    for kelas in individu:
        slot_harian = kelas[:-1]

        for hari_id, hari in enumerate(slot_harian):
            for slot_id, mapel in enumerate(hari):

                mgmp_hari = mapel_mgmp.get(mapel)

                if mgmp_hari is None:
                    continue

                if hari_id == mgmp_hari:
                    batas_slot = batas_mgmp[hari_id]

                    if slot_id > batas_slot:
                        pelanggaran += 1
                        cost += VIOLATION_COST
                        
    return pelanggaran, cost


In [231]:
for individu in populasi:
    pelanggaran, cost = mgmp_slot(individu, relasi, batas_mgmp,  VIOLATION_COST)
    print("Hard Constraint: ", pelanggaran)
    print("Cost: ", cost)

Hard Constraint:  133
Cost:  13300
Hard Constraint:  110
Cost:  11000
Hard Constraint:  120
Cost:  12000
Hard Constraint:  116
Cost:  11600
Hard Constraint:  127
Cost:  12700
Hard Constraint:  104
Cost:  10400
Hard Constraint:  109
Cost:  10900
Hard Constraint:  128
Cost:  12800
Hard Constraint:  132
Cost:  13200
Hard Constraint:  113
Cost:  11300
Hard Constraint:  109
Cost:  10900
Hard Constraint:  122
Cost:  12200
Hard Constraint:  127
Cost:  12700
Hard Constraint:  130
Cost:  13000
Hard Constraint:  122
Cost:  12200
Hard Constraint:  117
Cost:  11700
Hard Constraint:  115
Cost:  11500
Hard Constraint:  146
Cost:  14600
Hard Constraint:  124
Cost:  12400
Hard Constraint:  117
Cost:  11700
Hard Constraint:  122
Cost:  12200
Hard Constraint:  143
Cost:  14300
Hard Constraint:  136
Cost:  13600
Hard Constraint:  138
Cost:  13800
Hard Constraint:  115
Cost:  11500
Hard Constraint:  117
Cost:  11700
Hard Constraint:  114
Cost:  11400
Hard Constraint:  150
Cost:  15000
Hard Constraint:  14

### 6.7. Guru Wali kelas

# 7. Proses Seleksi